# Amazon ML 2026 — Master Colab
Independent, resumable baseline for SanskariXD/Amazon-ML-C-26. **Real score unknown until you run the official dataset.** CPU runtime is sufficient. Run cells in order; keep this notebook tied to a fixed commit while resuming. No portal submission is automated.

## 1. Mount Google Drive
Use the account containing your original dataset ZIP. Raw data is never modified.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
ROOT = Path('/content/drive/MyDrive/AmazonML2026')
ZIP_PATH = Path('/content/drive/MyDrive/6ab10eb3b23ba_student_resource.zip')
for name in ['data/raw','data/processed','checkpoints','candidates','features','models','experiments','outputs','logs','runs']:
    (ROOT/name).mkdir(parents=True, exist_ok=True)
assert ZIP_PATH.is_file(), f'ZIP not found: {ZIP_PATH}. Correct ZIP_PATH before continuing.'
# Verified resource ZIP size: 1,094,823,222 bytes.
# Use the mounted private file; no public sharing or re-upload is needed.

## 2. Detect runtime resources
Check your Drive account quota separately; free-space reported by the mount may not equal your remaining Drive quota.

In [ ]:
import os, shutil, subprocess
print('CPU cores:', os.cpu_count())
print(subprocess.run(['free','-h'],capture_output=True,text=True).stdout)
print('Local free GB:', round(shutil.disk_usage('/content').free/2**30,2))
try:
    print(subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],text=True))
except (OSError,subprocess.SubprocessError):
    print('No GPU detected; CPU baseline is supported.')

## 3. Clone or reuse GitHub code
The checkout lives on Colab local disk. Set PINNED_COMMIT to the hash from your first run when reconnecting; code changes intentionally produce a different artifact run.

In [ ]:
REPO = Path('/content/Amazon-ML-C-26')
PINNED_COMMIT = ''  # Optional explicit commit; otherwise resume the saved commit below.
SESSION_FILE=ROOT/'active_code_commit.json'
import json
if not PINNED_COMMIT and SESSION_FILE.exists():
    PINNED_COMMIT=json.loads(SESSION_FILE.read_text())['commit']
if not (REPO/'.git').exists():
    subprocess.run(['git','clone','https://github.com/SanskariXD/Amazon-ML-C-26.git',str(REPO)],check=True)
if not PINNED_COMMIT:
    if subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip():
        raise RuntimeError('Local repo has edits. Save them before switching to the latest code.')
    subprocess.run(['git','-C',str(REPO),'fetch','origin','main'],check=True)
    subprocess.run(['git','-C',str(REPO),'checkout','--detach','origin/main'],check=True)
if PINNED_COMMIT:
    subprocess.run(['git','-C',str(REPO),'fetch','origin',PINNED_COMMIT],check=True)
    subprocess.run(['git','-C',str(REPO),'checkout','--detach',PINNED_COMMIT],check=True)
os.chdir(REPO)
print('Code commit:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())
COMMIT=subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()
SESSION_FILE.write_text(json.dumps({'commit':COMMIT}))
# To deliberately upgrade later, set PINNED_COMMIT to the new commit.

## 4. Install pinned dependencies

In [ ]:
import sys
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'],check=True)
subprocess.run([sys.executable,'-m','unittest','discover','-s','tests','-v'],check=True)

## 5. Verify and extract the official dataset
Extraction resumes by file. The raw ZIP is hashed and kept unchanged. Allow enough space for ZIP + extracted data + indexes/features.

In [ ]:
subprocess.run([sys.executable,'-m','src.prepare','--zip',str(ZIP_PATH),'--destination',str(ROOT/'data/raw')],check=True)
import json
DATASET=Path(json.loads((ROOT/'data/raw/dataset_location.json').read_text())['dataset'])
print('Dataset:',DATASET)
assert (DATASET.parent/'utils/validate_submission.py').is_file(), 'Organizer validator missing from bundle'

## 6. Resume/checkpoint detection
Use baseline first. Select stage2.json only as a separate measured experiment. Each stage resolves the same code/config/data fingerprint.

In [ ]:
os.environ['ER_LOCAL_CACHE']='/content/amazon-ml-local-cache'
Path(os.environ['ER_LOCAL_CACHE']).mkdir(parents=True,exist_ok=True)
CONFIG='configs/baseline.json'
def run(stage):
    subprocess.run([sys.executable,'-m','src.pipeline','--dataset',str(DATASET),'--work',str(ROOT/'runs'),'--config',CONFIG,'--stage',stage],check=True)
def current_run():
    return Path(json.loads((ROOT/'runs/latest_run.json').read_text())['work'])
if (ROOT/'runs/latest_run.json').exists():
    print('Previous run:',json.loads((ROOT/'runs/latest_run.json').read_text()))

## 7. First real-data profile — run up to here first
Stream counts, country coverage, missing fields, file sizes and match-count distribution without constructing a normalized database or indexes. Then download the small report and upload it to ChatGPT. This lets us choose the resource budget before expensive stages. Full relational validation follows in section 8.

In [ ]:
run('profile')
print((current_run()/'dataset_profile_preliminary.json').read_text())
run('report')
from google.colab import files
files.download(str(current_run()/'review_report.zip'))

## 8. Validation creation
Stable entity hashing assigns fit/tune/development/final-holdout groups in SQLite. Validation queries always see the full target universe. No label-derived maps are learned. Shared labelled targets stop the run until component splitting is implemented.

In [ ]:
run('audit')
import sqlite3
paths=json.loads((current_run()/'artifact_paths.json').read_text())
with sqlite3.connect(Path(paths['prepared'])/'records.sqlite') as db:
    print(db.execute("SELECT part,country,COUNT(*) FROM records WHERE split='train' AND source=1 GROUP BY part,country").fetchall())

## 9. Blocking experiment setup
Default: three views × top 20, at most 60 candidates per S1. Vocabulary uses a label-free sample, while retrieval scans all targets. Country blocking is off. Start with the synthetic integrity test before the expensive real run.

In [ ]:
subprocess.run([sys.executable,'-m','src.smoke'],check=True)
print(Path(CONFIG).read_text())

## 10. Full-pool blocking benchmark
Build target indexes and measure retrieval on 256 fitting entities against **all training targets**. Compare top 10/20/30/50 per view, pair/entity recall, candidate counts, RAM and projected time/storage. This is a screening estimate, not a model score. No holdout entities are used.

In [ ]:
run('benchmark')
print((current_run()/'experiments/blocking_benchmark.json').read_text())
run('report')
from google.colab import files
files.download(str(current_run()/'review_report.zip'))

## 11. Pair feature generation
The next baseline command checkpoints candidates and 27 pair/retrieval features per query batch. It also trains, tunes and evaluates. Query feature shards include all model-scored candidates; no truth-positive injection occurs.

**First real-data run: pause here and upload review_report.zip to ChatGPT.** It contains profile/benchmark numbers, not raw business records. Use those measurements to decide candidate count and whether full execution fits the available time and disk.

## 12. Model training
Default: 15k fitting S1, all retrieved negatives, single LightGBM. The fitting stage checks a conservative RAM estimate. This is sampled fitting against a full target universe, not a full-data retrain.

In [ ]:
run('baseline')

## 13. Threshold/decision tuning
The threshold is chosen only on tuning entities. Development entities measure the chosen rule. Expected-F is an optional later config experiment.

In [ ]:
print((current_run()/'models/decision.json').read_text())
print((current_run()/'experiments/dev.json').read_text())

## 14. Error analysis
Review missed retrieval pairs before changing models. The stored sample includes false positives, false negatives, country and singleton status.

In [ ]:
errors=json.loads((current_run()/'experiments/dev_errors.json').read_text())
for example in errors[:15]:print(example)

## 15. Optional measured experiments
E002: configs/stage2.json uses inner-OOF query-context stacking; it reuses shared normalization/index/feature caches. Strict country transfer is also implemented: train and tune on one country, evaluate on another country’s development entities. Neither procedure opens the final holdout. Reverse retrieval, graph-wide exclusivity, ensembles and neural reranking await benchmark/error evidence.

In [ ]:
RUN_STAGE2=False
if RUN_STAGE2:
    CONFIG='configs/stage2.json'
    run('baseline')
RUN_COUNTRY_STRESS=False
if RUN_COUNTRY_STRESS:
    run('loco')
run('report')

## 16. Freeze, final holdout and test inference
Set the chosen CONFIG above before running. Flip the switch only after reviewing development metrics, available disk and projected time. This uses the selected fitted model; it does not silently retrain on all data. Once the holdout is inspected, do not tune to it.

In [ ]:
RUN_FINAL=False
if RUN_FINAL:
    run('holdout')
    run('infer')

## 17. Submission generation
Inference writes matching_results.tsv and candidate_pairs.tsv under the current run outputs folder. Every S1, including France and empty predictions, appears once. Final matches must be a subset of scored candidates.

In [ ]:
if RUN_FINAL:
    print('Output folder:',current_run()/'outputs')

## 18. Official validation
Inference already invokes the organizer script; this reruns it explicitly. No file is uploaded to the challenge portal by this notebook.

In [ ]:
if RUN_FINAL:
    run('validate')
    print((current_run()/'outputs/official_validation.json').read_text())

## 19. Save artifacts and final package
All expensive artifacts already persist in the current Drive run. Fill the organizer Documentation_template.md with your actual methodology/results, place it in the current run directory, then enable packaging. The ZIP includes runnable code/config, documentation and both validated outputs.

`runs/_shared/` stores reusable prepared data, indexes and features. Each run’s artifact_paths.json points to them. ER_LOCAL_CACHE holds disposable local copies for faster reads; persistent checkpoints remain in Drive. On reconnect, saved code commit and stage fingerprints preserve compatible work.

In [ ]:
BUILD_PACKAGE=False
if BUILD_PACKAGE:
    run('package')
print('Saved work:',current_run())